# hey Jillian wake-word training

Uses the user-approved **Jillian, the name** pronunciation with the full prefix **hey**. Choose the **2026.04 CPU** Colab runtime and run the cells in order. Training runs in an isolated environment. The final cell checks the ONNX export and provides its download. Test held-out recordings and the physical device before switching the active detector.


In [ ]:
# @title 1. Set up the isolated environment and hear hey Jillian
import os, sys, subprocess
from pathlib import Path

def run_visible(command, env=None):
    with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1) as child:
        try:
            for line in child.stdout:
                print(line, end='', flush=True)
        except BaseException:
            child.kill()
            child.wait()
            raise
        if child.wait():
            raise subprocess.CalledProcessError(child.returncode, command)
VENV = Path('/content/gilligan-venv')
if not (VENV / 'bin/pip').exists():
    run_visible([sys.executable, '-m', 'pip', 'install', 'virtualenv'])
    run_visible([sys.executable, '-m', 'virtualenv', '--system-site-packages', str(VENV)])
constraints = Path('/content/gilligan-constraints.txt')
constraints.write_text('numpy==1.26.4\nsetuptools<81\npyarrow==14.0.2\nml-dtypes==0.5.1\n')
training_env = os.environ.copy()
training_env['PATH'] = str(VENV / 'bin') + ':' + training_env['PATH']
training_env['PIP_CONSTRAINT'] = str(constraints)
training_python = str(VENV / 'bin/python')
run_visible([training_python, '-m', 'pip', 'install', 'numpy==1.26.4', 'setuptools<81'], env=training_env)
run_visible([training_python, '-m', 'pip', 'install', 'torch==2.5.0', 'torchvision==0.20.0', 'torchaudio==2.5.0', '--index-url', 'https://download.pytorch.org/whl/cpu'], env=training_env)
run_visible([training_python, '-m', 'pip', 'install', 'piper-tts', 'piper-phonemize-cross', 'webrtcvad'], env=training_env)
Path('/content/gilligan_setup.ipy').write_text('# @title  { display-mode: "form" }\n# @markdown # 1. Test Example Training Clip Generation\n# @markdown Since openWakeWord models are trained on synthetic examples of your\n# @markdown target wake word, it\'s a good idea to make sure that the examples\n# @markdown sound correct. Type in your target wake word below, and run the\n# @markdown cell to listen to it.\n# @markdown\n# @markdown Here are some tips that can help get the wake word to sound right:\n\n# @markdown - If your wake word isn\'t being pronounced in the way\n# @markdown you want, try spelling out the sounds phonetically with underscores\n# @markdown separating each part.\n# @markdown For example: "hey siri" --> "hey_seer_e".\n\n# @markdown - Spell out numbers ("2" --> "two")\n\n# @markdown - Avoid all punctuation except for "?" and "!", and remove unicode characters\n\nimport sys\nprint("TRAINING_PYTHON", sys.version)\nimport subprocess\nimport os\nimport sys\nfrom IPython.display import Audio\n\nif not os.path.exists("./piper-sample-generator"):\n    !git clone https://github.com/rhasspy/piper-sample-generator\n    !wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt \'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt\'\n    !cd piper-sample-generator && git checkout 213d4d5\n\n    # Install system dependencies\n    !pip install piper-tts piper-phonemize-cross\n    !pip install webrtcvad\n    # !pip install \'torch<=2.5\' torchvision torchaudio\n    !pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cpu\n\ntarget_word = \'hey Jillian\' # @param {type:"string"}\n\nif "piper-sample-generator/" not in sys.path:\n    sys.path.append("piper-sample-generator/")\nfrom generate_samples import generate_samples\n\ndef text_to_speech(text):\n    generate_samples(text = text,\n                max_samples=1,\n                length_scales=[1.1],\n                noise_scales=[0.7], noise_scale_ws = [0.7],\n                output_dir = \'./\', batch_size=1, auto_reduce_batch_size=True,\n                file_names=["test_generation.wav"]\n                )\n\ntext_to_speech(target_word)\n\nfrom IPython.display import Audio\nassert os.path.isfile("test_generation.wav")\nprint("SAMPLE_READY")\n\n')
run_visible([training_python, '-m', 'IPython', '/content/gilligan_setup.ipy'], env=training_env)
from IPython.display import Audio, display
display(Audio('test_generation.wav'))


In [ ]:
Path('/content/gilligan_data.ipy').write_text('# @title  { display-mode: "form" }\n# @markdown # 2. Download Data\n# @markdown Training custom models requires downloading a wide variety of data\n# @markdown that will help make the model perform well in real-world scenarios.\n# @markdown This example notebook will download small samples of background noise,\n# @markdown music, and Room Impulse Responses (to add echo). This will still produce\n# @markdown a custom model that performs well, but if you are interested in adding even more,\n# @markdown feel free to extend this notebook to download the full datasets and even add\n# @markdown your own!\n# @markdown\n# @markdown Downloading this example data will usually take about 15 minutes.\n\n# @markdown **Important note!** The data downloaded here has a mixture of different\n# @markdown licenses and usage restrictions. As such, any custom models trained with this\n# @markdown data should be considered as appropriate for **non-commercial** personal use only.\n\n# ## Install all dependencies\n# !pip install datasets\n# !pip install scipy\n# !pip install tqdm\n\nimport locale\ndef getpreferredencoding(do_setlocale = True):\n    return "UTF-8"\nlocale.getpreferredencoding = getpreferredencoding\n\n# Keep all pip commands on one NumPy/Arrow ABI throughout this runtime.\nfrom pathlib import Path\nPath(\'/content/gilligan-constraints.txt\').write_text(\'numpy==1.26.4\\nsetuptools<81\\npyarrow==14.0.2\\nml-dtypes==0.5.1\\n\')\n\n# install openwakeword (full installation to support training)\n!git clone https://github.com/dscripka/openwakeword\n!pip install -c /content/gilligan-constraints.txt -e ./openwakeword --no-deps\n# !cd openwakeword\n\n# install other dependencies\n!pip install -c /content/gilligan-constraints.txt mutagen==1.47.0\n!pip install -c /content/gilligan-constraints.txt torchinfo==1.8.0\n!pip install -c /content/gilligan-constraints.txt torchmetrics==1.2.0\n!pip install -c /content/gilligan-constraints.txt speechbrain==0.5.14\n!pip install -c /content/gilligan-constraints.txt audiomentations==0.33.0\n!pip install -c /content/gilligan-constraints.txt torch-audiomentations==0.11.0\n!pip install -c /content/gilligan-constraints.txt acoustics==0.2.6\n# !pip uninstall tensorflow -y\n# !pip install tensorflow-cpu==2.8.1\n# !pip install protobuf==3.20.3\n# !pip install tensorflow_probability==0.16.0\n# !pip install onnx_tf==1.10.0\n!pip install -c /content/gilligan-constraints.txt onnxruntime==1.22.1 onnx==1.19.1\n# !pip install ai_edge_litert==1.2.0\n!pip install -c /content/gilligan-constraints.txt pronouncing==0.2.0\n!pip install -c /content/gilligan-constraints.txt datasets==2.14.6\n!pip install -c /content/gilligan-constraints.txt deep-phonemizer==0.0.19\n\n# Python 3.11 / ONNX-only compatibility for this Pi target.\n!pip install -c /content/gilligan-constraints.txt numpy==1.26.4 \'setuptools<81\' pyarrow==14.0.2 ml-dtypes==0.5.1\n\n# Download required models (workaround for Colab)\nimport os\nos.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)\n!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx\n!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite\n!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx\n!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite\n\n# Imports\nimport sys\n\nif "piper-sample-generator/" not in sys.path:\n    sys.path.append("piper-sample-generator/")\nfrom generate_samples import generate_samples\n\nimport numpy as np\nimport torch\nimport sys\nfrom pathlib import Path\nimport uuid\nimport yaml\nimport datasets\nimport scipy\nfrom tqdm import tqdm\n\n## Download all data\n\n## Download MIR RIR data (takes about ~2 minutes)\noutput_dir = "./mit_rirs"\nif not os.path.exists(output_dir):\n    os.mkdir(output_dir)\n    !git lfs install\n    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses\n    rir_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("./MIT_environmental_impulse_responses/16khz").glob("*.wav")]}).cast_column("audio", datasets.Audio())\n    # Save clips to 16-bit PCM wav files\n    for row in tqdm(rir_dataset):\n        name = row[\'audio\'][\'path\'].split(\'/\')[-1]\n        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row[\'audio\'][\'array\']*32767).astype(np.int16))\n\n## Download noise and background audio (takes about ~3 minutes)\n\n# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)\n# Download one part of the audioset .tar files, extract, and convert to 16khz\n# For full-scale training, it\'s recommended to download the entire dataset from\n# https://huggingface.co/datasets/agkphysics/AudioSet, and\n# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)\n\nif not list(Path("audioset_16k").glob("*.wav")):\n    os.makedirs("audioset",exist_ok=True)\n\n    fname = "bal_train09.tar"\n    out_dir = f"audioset/{fname}"\n    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/5a2fa42a1506470d275a47ff8e1fdac5b364e6ef/data/" + fname\n    import subprocess\n    subprocess.run(["wget","-q","-O",out_dir,link],check=True)\n    !cd audioset && tar -xf bal_train09.tar\n\n    output_dir = "./audioset_16k"\n    if not os.path.exists(output_dir):\n        os.mkdir(output_dir)\n\n    # Save clips to 16-bit PCM wav files\n    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})\n    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))\n    for row in tqdm(audioset_dataset):\n        name = row[\'audio\'][\'path\'].split(\'/\')[-1].replace(".flac", ".wav")\n        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row[\'audio\'][\'array\']*32767).astype(np.int16))\n\n# Free Music Archive dataset\n# https://github.com/mdeff/fma\n\noutput_dir = "./fma"\nif not os.path.exists(output_dir):\n    os.mkdir(output_dir)\n    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)\n    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))\n\n    # Save clips to 16-bit PCM wav files\n    n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training\n    for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips\n        row = next(fma_dataset)\n        name = row[\'audio\'][\'path\'].split(\'/\')[-1].replace(".mp3", ".wav")\n        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row[\'audio\'][\'array\']*32767).astype(np.int16))\n        i += 1\n        if i == n_hours*3600//30:\n            break\n\n# Download pre-computed openWakeWord features for training and validation\n\n# training set (~2,000 hours from the ACAV100M Dataset)\n# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information\nif not Path(\'openwakeword_features_ACAV100M_2000_hrs_16bit.npy\').exists() or Path(\'openwakeword_features_ACAV100M_2000_hrs_16bit.npy\').stat().st_size != 17280000128:\n    print(\'Downloading/resuming negative features (16 GB)\',flush=True)\n    !wget -q -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy\nassert Path(\'openwakeword_features_ACAV100M_2000_hrs_16bit.npy\').stat().st_size == 17280000128\n\n# validation set for false positive rate estimation (~11 hours)\nif not Path(\'validation_set_features.npy\').exists() or Path(\'validation_set_features.npy\').stat().st_size != 184836608:\n    !wget -q -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy\nassert Path(\'validation_set_features.npy\').stat().st_size == 184836608\n\n\nassert list(Path("audioset_16k").glob("*.wav")), "AudioSet backgrounds missing"\nassert list(Path("fma").glob("*.wav")), "FMA backgrounds missing"\nprint("DATA_READY",len(list(Path("audioset_16k").glob("*.wav"))),len(list(Path("fma").glob("*.wav"))))\n')
run_visible([training_python,'-m','IPython','/content/gilligan_data.ipy'],env=training_env)


In [ ]:
# @title Train hey Jillian with the approved Jillian-name pronunciation
from pathlib import Path
import os,subprocess
training_python='/content/gilligan-venv/bin/python'
training_env=os.environ.copy();training_env['PATH']='/content/gilligan-venv/bin:'+training_env['PATH']
Path('/content/hey_jillian_train.py').write_text('import sys,subprocess\nfrom pathlib import Path\ntarget_word=\'hey Jillian\'\ndef run_stage(stage):\n    command=[sys.executable,\'openwakeword/openwakeword/train.py\',\'--training_config\',\'hey_jillian_model.yaml\',\'--\'+stage]\n    log=[]\n    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors=\'replace\',bufsize=1) as child:\n        try:\n            for line in child.stdout:\n                print(line,end=\'\',flush=True);log.append(line)\n        except BaseException:\n            child.kill();child.wait();raise\n        if child.wait():raise subprocess.CalledProcessError(child.returncode,command)\n    return \'\'.join(log)\n\n# Stream identical overlapping false-positive windows instead of one enormous batch.\nfrom pathlib import Path\nPath(\'/content/openwakeword/openwakeword/bounded_validation.py\').write_text(\'"""Stream the trainer\\\'s original overlapping validation windows without expanding them."""\\nimport numpy as np\\n\\nclass FalsePositiveWindows:\\n    def __init__(self,path,length):\\n        self.features=np.load(path,mmap_mode=\\\'r\\\')\\n        self.length=int(length)\\n        if self.length<=0 or self.features.ndim!=2:\\n            raise ValueError(\\\'Invalid validation shape\\\')\\n    def __len__(self):\\n        # Match the original range(0, feature_count-window_length, 1) exactly.\\n        return max(0,len(self.features)-self.length)\\n    def __getitem__(self,index):\\n        if not 0<=index<len(self):raise IndexError(index)\\n        return np.array(self.features[index:index+self.length],copy=True),np.float32(0)\\n\\ndef false_positive_loader(path,length,batch_size=1024):\\n    import torch\\n    return torch.utils.data.DataLoader(FalsePositiveWindows(path,length),\\n                                       batch_size=batch_size,num_workers=0)\\n\')\ntrainer=Path(\'/content/openwakeword/openwakeword/train.py\')\ncode=trainer.read_text()\nif \'from bounded_validation import false_positive_loader\' not in code:\n    begin=code.index(\'        X_val_fp = np.load(config["false_positive_validation_data_path"])\')\n    end=code.index(\'        X_val_pos =\',begin)\n    code=code[:begin]+\'        from bounded_validation import false_positive_loader\\n        X_val_fp = false_positive_loader(config["false_positive_validation_data_path"], input_shape[0], batch_size=1024)\\n\\n\'+code[end:]\n    trainer.write_text(code)\n\n\n\n\n\n\n# Load default YAML config file for training\nimport yaml\nconfig = yaml.load(open("openwakeword/examples/custom_model.yml", \'r\').read(), yaml.Loader)\n\n# Modify values in the config and save a new version\nnumber_of_examples = 1000\nnumber_of_training_steps = 10000\nfalse_activation_penalty = 1500\nconfig["target_phrase"] = [target_word]\nconfig["model_name"] = "hey_jillian"\nconfig["n_samples"] = number_of_examples\nconfig["n_samples_val"] = max(500, number_of_examples//10)\nconfig["steps"] = number_of_training_steps\nconfig["target_accuracy"] = 0.5\nconfig["target_recall"] = 0.25\nconfig["output_dir"] = "./hey_jillian_model"\nconfig["max_negative_weight"] = false_activation_penalty\n\nconfig["custom_negative_phrases"] = ["Jillian", "hey Gilligan", "Gilligan", "hey Julian", "hey Jarvis"]\nconfig["background_paths"] = [\'./audioset_16k\', \'./fma\']  # multiple background datasets are supported\nconfig["false_positive_validation_data_path"] = "validation_set_features.npy"\nconfig["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}\n\nwith open(\'hey_jillian_model.yaml\', \'w\') as file:\n    documents = yaml.dump(config, file)\n\n# Fix upstream argparse string defaults so optional TFLite conversion stays off.\nfrom pathlib import Path\ntraining_source=Path(\'openwakeword/openwakeword/train.py\')\nsource=training_source.read_text()\nif \'default="False"\' in source:\n    training_source.write_text(source.replace(\'default="False"\',\'default=False\'))\n\n# Require complete background data before generating clips.\nfrom pathlib import Path\nassert list(Path(\'audioset_16k\').glob(\'*.wav\')), \'AudioSet backgrounds missing\'\nassert list(Path(\'fma\').glob(\'*.wav\')), \'FMA backgrounds missing\'\nimport subprocess\n\n# Generate clips\n# Verify the approved Jillian-name pronunciation before training.\nsys.path.insert(0,\'/content/piper-sample-generator\')\nfrom generate_samples import phonemize_espeak\npronunciation=phonemize_espeak(target_word,\'en-us\')\nprint(\'APPROVED_NAME_PRONUNCIATION\',pronunciation,flush=True)\nassert \'dʒ\' in \'\'.join(\'\'.join(sentence) for sentence in pronunciation), \'Expected soft J in Jillian\'\n\nrun_stage("generate_clips")\n\n# Step 2: Augment the generated clips\n\nrun_stage("augment_clips")\n\n# Step 3: Train model\n\n# Verify bounded validation with the actual PyTorch loader before training.\nfrom openwakeword.bounded_validation import false_positive_loader\nimport numpy as np\nshape=np.load(\'hey_jillian_model/hey_jillian/positive_features_test.npy\').shape[1:]\nvalidation=false_positive_loader(config[\'false_positive_validation_data_path\'],shape[0],batch_size=1024)\nx_check,y_check=next(iter(validation))\nassert x_check.shape[0]<=1024 and x_check.shape[1:]==shape\nassert not y_check.any()\nprint(\'BOUNDED_VALIDATION_READY\',len(validation.dataset),\'windows; batch\',tuple(x_check.shape),flush=True)\ndel x_check,y_check,validation\ntraining_log=run_stage("train_model")\n\n\n\nfrom pathlib import Path\nimport hashlib,json,re\nmodel_path=Path(\'hey_jillian_model/hey_jillian.onnx\')\nassert model_path.is_file(), \'Training did not export an ONNX model\'\nmetrics={}\nfor key,label in [(\'accuracy\',\'Accuracy\'),(\'recall\',\'Recall\'),(\'false_positives_per_hour\',\'False Positives per Hour\')]:\n    match=re.search(r\'Final Model \'+label+r\': ([0-9.eE+-]+)\',training_log)\n    if match:metrics[key]=float(match.group(1))\nmanifest={\'phrase\':\'hey Jillian\',\'training_text\':target_word,\'pronunciation_phonemes\':pronunciation,\'model_name\':\'hey_jillian\',\'sha256\':hashlib.sha256(model_path.read_bytes()).hexdigest(),\'positive_training_examples\':number_of_examples,\'positive_validation_examples\':config[\'n_samples_val\'],\'initial_training_steps\':number_of_training_steps,\'follow_up_steps\':[1000,1000],\'validation\':metrics}\nmodel_path.with_suffix(\'.json\').write_text(json.dumps(manifest,indent=2))\nprint(\'MODEL_READY\',model_path,model_path.stat().st_size,manifest[\'sha256\'],flush=True)\n')
with subprocess.Popen([training_python,'/content/hey_jillian_train.py'],env=training_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace',bufsize=1) as child:
    try:
        for line in child.stdout:print(line,end='',flush=True)
    except BaseException:
        child.kill();child.wait();raise
    if child.wait():raise subprocess.CalledProcessError(child.returncode,child.args)


In [ ]:
# @title Export the completed hey Jillian wake model
from pathlib import Path
import base64,hashlib,json,subprocess,zipfile
from IPython.display import HTML,display
model=Path('/content/hey_jillian_model/hey_jillian.onnx')
manifest_path=model.with_suffix('.json')
assert model.is_file() and manifest_path.is_file(),'Training has not completed'
manifest=json.loads(manifest_path.read_text())
assert hashlib.sha256(model.read_bytes()).hexdigest()==manifest['sha256']
result=subprocess.run(['/content/gilligan-venv/bin/python','-c','import onnx,sys;onnx.checker.check_model(sys.argv[1]);print("ONNX_VALID")',str(model)],text=True,capture_output=True,check=True)
print(result.stdout)
manifest['openwakeword_revision']=subprocess.check_output(['git','-C','/content/openwakeword','rev-parse','HEAD'],text=True).strip()
manifest['piper_revision']=subprocess.check_output(['git','-C','/content/piper-sample-generator','rev-parse','HEAD'],text=True).strip()
manifest['piper_source_sha256']=hashlib.sha256(Path('/content/piper-sample-generator/generate_samples.py').read_bytes()).hexdigest()
manifest_path.write_text(json.dumps(manifest,indent=2))
archive=Path('/content/hey-Jillian-wake-model.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(model,model.name);bundle.write(manifest_path,manifest_path.name)
encoded=base64.b64encode(archive.read_bytes()).decode('ascii')
display(HTML('<section style="background:#111;color:white;padding:32px;font:24px/1.5 sans-serif"><h2 style="font-size:36px">hey Jillian model ready</h2><p>Jillian, the name—approved pronunciation.</p><a style="display:inline-block;background:white;color:black;padding:20px;min-height:44px;font-size:26px;font-weight:bold" download="hey-Jillian-wake-model.zip" href="data:application/zip;base64,'+encoded+'">Download hey Jillian model</a><p style="overflow-wrap:anywhere">SHA-256: '+manifest['sha256']+'</p></section>'))
